# Un solo registro por inmueble (E8)

**Pregunta:** si contamos cada inmueble una sola vez —aunque esté en FincaRaíz y en Metrocuadrado, o lo
publiquen varias inmobiliarias en el mismo portal—, ¿cuánta oferta de vivienda en venta hay de verdad?

**Datos:** los avisos de vivienda activos (apartamento, apartaestudio y casa) de los dos portales, según el
estado del barrido del 29-oct-2026 (`estado/` y `estado_m2_venta/`).

**Método** ([src/unificar.py](../src/unificar.py), con las señales de [src/deduplicar.py](../src/deduplicar.py)):
1. **Candidatos:** mismo tipo, a menos de 500 m, área parecida (±15%, la mejor combinación de área total y
   construida) y precio parecido (±30%). Nunca dos avisos del mismo anunciante en el mismo portal.
2. **Puntaje** de cada par: texto (coseno de fragmentos de caracteres), ficha (habitaciones y baños topados en 5:
   Metrocuadrado publica «5 o más»), distancia y precio; piso distinto como veto, y habitaciones y baños distintos
   a la vez como resta. El mismo puntaje del análisis de traslape, con el texto calculado por *hashing* para que alcance
   la memoria con todo el país (en Cali acepta el 98,8% de los mismos pares).
3. **Grupos:** se unen los pares de puntaje ≥ 0,65, de mayor a menor, solo si **todos** los avisos del grupo
   se parecen entre sí (≥ 0,5), sin dos avisos del mismo anunciante y portal, sin pisos distintos y con un
   máximo de 6 avisos.

Solo cifras agregadas: los avisos concretos no se publican aquí.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE))
pct = lambda x: f"{x:.0%}"
mil = lambda x: f"{x:,.0f}".replace(",", ".")
av = pd.read_parquet(BASE / "data/unificado/avisos.parquet")
inm = pd.read_parquet(BASE / "data/unificado/inmuebles.parquet")
multi = inm[inm["n_avisos"] > 1]
pd.Series({
    "avisos de vivienda activos": mil(len(av)),
    "  de FincaRaíz": mil((av.portal == "fincaraiz").sum()),
    "  de Metrocuadrado": mil((av.portal == "metrocuadrado").sum()),
    "inmuebles únicos": mil(len(inm)),
    "avisos que sobran (mismo inmueble)": pct(1 - len(inm) / len(av)),
    "inmuebles con 2 o más avisos": mil(len(multi)),
    "  en los dos portales": mil((multi.portales == "fincaraiz+metrocuadrado").sum()),
    "  repetidos solo en FincaRaíz": mil((multi.portales == "fincaraiz").sum()),
    "  repetidos solo en Metrocuadrado": mil((multi.portales == "metrocuadrado").sum()),
}).to_frame("todo el país")

,todo el país
avisos de vivienda activos,273.404
de FincaRaíz,131.695
de Metrocuadrado,141.709
inmuebles únicos,203.873
avisos que sobran (mismo inmueble),25%
inmuebles con 2 o más avisos,55.030
en los dos portales,50.998
repetidos solo en FincaRaíz,2.213
repetidos solo en Metrocuadrado,1.819


## 1. Por ciudad

`aumento frente a FincaRaíz`: cuántos inmuebles más hay sumando Metrocuadrado, frente a los inmuebles únicos que
ya tiene FincaRaíz (también sin repetidos).

In [2]:
av["ciudad_n"] = av["ciudad"].fillna("").str.replace(r",.*", "", regex=True).str.replace(" D.C.", "", regex=False).str.strip()
t = av.groupby("ciudad_n").agg(avisos=("clave", "size"),
                               fincaraiz=("portal", lambda s: (s == "fincaraiz").sum()),
                               metrocuadrado=("portal", lambda s: (s == "metrocuadrado").sum()),
                               inmuebles=("id_unificado", "nunique"))
t["sobran"] = 1 - t["inmuebles"] / t["avisos"]
t["inmuebles solo FincaRaíz"] = av[av.portal == "fincaraiz"].groupby("ciudad_n")["id_unificado"].nunique()
t["aumento frente a FincaRaíz"] = t["inmuebles"] / t["inmuebles solo FincaRaíz"] - 1
t.sort_values("avisos", ascending=False).head(15).rename_axis("ciudad").style.format(
    {"avisos": mil, "fincaraiz": mil, "metrocuadrado": mil, "inmuebles": mil, "inmuebles solo FincaRaíz": mil,
     "sobran": pct, "aumento frente a FincaRaíz": lambda v: f"+{v:.0%}"})

,avisos,fincaraiz,metrocuadrado,inmuebles,sobran,inmuebles solo FincaRaíz,aumento frente a FincaRaíz
ciudad,,,,,,,
Bogotá,74.053,32.319,41.734,51.616,30%,28.622,+80%
Medellín,34.549,16.197,18.352,26.598,23%,15.227,+75%
Cali,18.713,10.022,8.691,14.444,23%,9.459,+53%
Barranquilla,14.858,7.573,7.285,10.261,31%,7.193,+43%
Envigado,11.528,6.033,5.495,9.155,21%,5.639,+62%
Pereira,9.189,3.539,5.650,7.306,20%,3.452,+112%
Rionegro,6.744,3.224,3.520,4.915,27%,3.036,+62%
Manizales,6.623,4.128,2.495,4.644,30%,3.573,+30%
Chía,6.087,2.794,3.293,4.427,27%,2.582,+71%


## 2. Cómo son los grupos

In [3]:
tam = inm["n_avisos"].value_counts().sort_index().rename("inmuebles").to_frame()
tam["avisos"] = tam.index * tam["inmuebles"]
tam.rename_axis("avisos por inmueble").style.format(mil)

,inmuebles,avisos
avisos por inmueble,,
1,148.843,148.843
2,46.994,93.988
3,4.304,12.912
4,1.900,7.600
5,931,4.655
6,901,5.406


In [4]:
dif = multi["precio_max"] / multi["precio_min"] - 1
pd.Series({
    "precio idéntico en todos sus avisos": pct(np.mean(dif < 0.005)),
    "difieren menos de 5%": pct(np.mean((dif >= 0.005) & (dif < 0.05))),
    "difieren 5% o más": pct(np.mean(dif >= 0.05)),
}).to_frame("inmuebles con 2+ avisos")

,inmuebles con 2+ avisos
precio idéntico en todos sus avisos,91%
difieren menos de 5%,6%
difieren 5% o más,3%


## 3. Conclusiones

1. **Uno de cada cuatro avisos sobra:** de ~273.000 avisos de vivienda activos en los dos portales quedan
   ~204.000 inmuebles únicos (−25%). Casi todo el traslape es entre portales; los repetidos dentro de un mismo
   portal (varias inmobiliarias) son ~4.000 inmuebles, y otros ~7.600 están repetidos en un portal y además en el
   otro.
2. **Metrocuadrado suma mucha oferta en todas las ciudades grandes:** frente a FincaRaíz sola (sin repetidos),
   +80% en Bogotá, +75% en Medellín, +53% en Cali, +43% en Barranquilla.
3. **Dentro de un grupo el precio casi siempre es el mismo** (90%). Las diferencias son las que interesan: el
   mismo inmueble más barato en otro portal o con otra inmobiliaria.
4. **Corre rápido:** todo el país en ~2 min y ~7 GB de memoria (6,5 millones de pares candidatos).

**Pendiente (BACKLOG E8):**
- **Revisión a mano** de una muestra más grande (ojalá con los agentes) para fijar los umbrales con números. La
  primera (29-oct, 8 grupos de Cali) dio 7 de 8 bien; el que no era una casa con la misma área y precio pero otra
  ficha, y de ahí la resta por habitaciones y baños distintos. Siguen dudosos los apartamentos con la misma ficha
  en conjuntos vecinos, con textos distintos y precios casi iguales.
- **Coordenadas genéricas:** cientos de avisos comparten el mismo punto (el centro de la ciudad o de un sector,
  cuando el anunciante no ubica el inmueble); para ellos la distancia no dice nada.
- **`id_unificado` estable** de un día a otro y **estado unificado** que usen la alerta (comparables sin
  repetidos, avisos de los dos portales), el modelo y el mapa.
- **Proyectos:** varias unidades del mismo tipo y precio en un proyecto pueden quedar juntas; para comparables
  da igual, para contar oferta las subestima.